# CLIP4CAD-GFA v4 Evaluation

This notebook evaluates the trained GFA v4 model with **Slot Attention and Query Distillation**.

## Evaluation Metrics
1. **Retrieval metrics**: Text→BRep, Text→PC, PC→BRep R@K
2. **Self-grounding quality**: Cosine similarity between guided and self embeddings
3. **Query alignment**: Cosine similarity between T_feat and Q_self (KEY METRIC!)
4. **Text-free inference**: Compare self-grounding retrieval with text-guided

## Expected Results (v4 with Query Distillation)
| Metric | v2.4 (Broken) | v4 Target |
|--------|---------------|-------------|
| Query alignment | N/A | **≥0.70** |
| Text→BRep R@1 (guided) | 71.5% | ≥65% |
| Text→BRep R@1 (self) | 0.05% | **≥55%** |
| Self cosine BRep | 0.08 | **≥0.80** |
| Gap (guided - self) | 71.5% | **<10%** |

In [ ]:
# Cell 1: Imports
import sys
sys.path.insert(0, '..')

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# Cell 2: Load Model

from clip4cad.models import CLIP4CAD_GFA_v4, GFAv4Config

# Path to trained model
CHECKPOINT_PATH = Path("../outputs/gfa_v4/checkpoint_best.pt")

# Create model with same config as training
config = GFAv4Config(
    d_face=48,
    d_edge=12,
    d_pc=1024,
    d_text=3072,
    d_unified=256,
    d_proj=128,
    d_ground=128,
    num_slots=12,
    num_detail_queries=8,
    num_slot_iterations=3,
    slot_hidden_dim=512,
)

model = CLIP4CAD_GFA_v4(config).to(device)

# Load weights
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"Loaded model from {CHECKPOINT_PATH}")
print(f"Epoch: {checkpoint.get('epoch', 'N/A')}")
print(f"Best self-cosine: {checkpoint.get('best_self_cosine', 'N/A')}")
print(f"Best query-align: {checkpoint.get('best_query_align', 'N/A')}")

In [ ]:
# Cell 3: Load Data using GFAMappedDataset

from clip4cad.data.gfa_dataset import GFAMappedDataset

DATA_ROOT = Path("d:/Defect_Det/MMCAD/data")
PC_FILE = Path("c:/Users/User/Desktop/pc_embeddings_full.h5")
BREP_FILE = Path("c:/Users/User/Desktop/brep_features.h5")
TEXT_FILE = Path("c:/Users/User/Desktop/text_embeddings.h5")

print("Loading validation dataset...")

val_dataset = GFAMappedDataset(
    data_root=str(DATA_ROOT),
    split="val",
    pc_file=str(PC_FILE),
    text_file=str(TEXT_FILE),
    brep_file=str(BREP_FILE),
    num_rotations=1,
    load_to_memory=False,
    use_live_text=False,
)

print(f"Validation samples: {len(val_dataset)}")

In [ ]:
# Cell 4: Encode All Samples

from clip4cad.data.gfa_dataset import gfa_collate_fn

@torch.no_grad()
def encode_all_samples(model, dataset, batch_size=64):
    """
    Encode all samples and return embeddings plus query features.
    """
    model.eval()
    
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        collate_fn=gfa_collate_fn,
    )
    
    # Embeddings
    z_brep_guided_all = []
    z_pc_guided_all = []
    z_text_all = []
    z_brep_self_all = []
    z_pc_self_all = []
    
    # Query features (for query alignment analysis)
    T_feat_all = []
    Q_brep_self_all = []
    Q_pc_self_all = []
    conf_text_all = []
    
    for batch in tqdm(loader, desc="Encoding"):
        outputs = model(batch)
        
        # Embeddings
        z_brep_guided_all.append(F.normalize(outputs['z_brep'], dim=-1).cpu())
        z_pc_guided_all.append(F.normalize(outputs['z_pc'], dim=-1).cpu())
        z_text_all.append(F.normalize(outputs['z_text'], dim=-1).cpu())
        z_brep_self_all.append(F.normalize(outputs['z_brep_self'], dim=-1).cpu())
        z_pc_self_all.append(F.normalize(outputs['z_pc_self'], dim=-1).cpu())
        
        # Query features
        if 'T_feat' in outputs:
            T_feat_all.append(outputs['T_feat'].cpu())
        if 'Q_brep_self' in outputs:
            Q_brep_self_all.append(outputs['Q_brep_self'].cpu())
        if 'Q_pc_self' in outputs:
            Q_pc_self_all.append(outputs['Q_pc_self'].cpu())
        if 'confidence' in outputs:
            conf_text_all.append(outputs['confidence'].cpu())
    
    results = {
        'z_brep_guided': torch.cat(z_brep_guided_all),
        'z_pc_guided': torch.cat(z_pc_guided_all),
        'z_text': torch.cat(z_text_all),
        'z_brep_self': torch.cat(z_brep_self_all),
        'z_pc_self': torch.cat(z_pc_self_all),
    }
    
    # Add query features if available
    if T_feat_all:
        results['T_feat'] = torch.cat(T_feat_all)
    if Q_brep_self_all:
        results['Q_brep_self'] = torch.cat(Q_brep_self_all)
    if Q_pc_self_all:
        results['Q_pc_self'] = torch.cat(Q_pc_self_all)
    if conf_text_all:
        results['conf_text'] = torch.cat(conf_text_all)
    
    return results

# Encode all samples
print("Encoding validation set...")
embeddings = encode_all_samples(model, val_dataset)

print(f"\nEmbedding shapes:")
print(f"  z_brep_guided: {embeddings['z_brep_guided'].shape}")
print(f"  z_brep_self: {embeddings['z_brep_self'].shape}")
print(f"  z_pc_guided: {embeddings['z_pc_guided'].shape}")
print(f"  z_text: {embeddings['z_text'].shape}")
if 'T_feat' in embeddings:
    print(f"  T_feat: {embeddings['T_feat'].shape}")
if 'Q_brep_self' in embeddings:
    print(f"  Q_brep_self: {embeddings['Q_brep_self'].shape}")

In [ ]:
# Cell 5: Compute Retrieval Metrics

def compute_retrieval_metrics(query_emb, gallery_emb, k_values=[1, 5, 10]):
    """Compute R@K retrieval metrics."""
    sim = query_emb @ gallery_emb.T
    rankings = torch.argsort(sim, dim=1, descending=True)
    N = query_emb.shape[0]
    labels = torch.arange(N)
    
    results = {}
    for k in k_values:
        top_k = rankings[:, :k]
        correct = (top_k == labels.unsqueeze(1)).any(dim=1)
        results[f'R@{k}'] = correct.float().mean().item() * 100
    
    ranks = (rankings == labels.unsqueeze(1)).nonzero()[:, 1] + 1
    results['MRR'] = (1.0 / ranks.float()).mean().item() * 100
    
    return results

# Extract embeddings
z_brep_guided = embeddings['z_brep_guided']
z_pc_guided = embeddings['z_pc_guided']
z_text = embeddings['z_text']
z_brep_self = embeddings['z_brep_self']
z_pc_self = embeddings['z_pc_self']

print("=" * 70)
print("RETRIEVAL RESULTS (Text-Guided)")
print("=" * 70)

# Text → BRep (Guided)
text_brep_guided = compute_retrieval_metrics(z_text, z_brep_guided)
print(f"\nText → BRep (Guided):")
for k, v in text_brep_guided.items():
    print(f"  {k}: {v:.2f}%")

# Text → PC (Guided)
text_pc_guided = compute_retrieval_metrics(z_text, z_pc_guided)
print(f"\nText → PC (Guided):")
for k, v in text_pc_guided.items():
    print(f"  {k}: {v:.2f}%")

# PC → BRep (Guided)
pc_brep_guided = compute_retrieval_metrics(z_pc_guided, z_brep_guided)
print(f"\nPC → BRep (Guided):")
for k, v in pc_brep_guided.items():
    print(f"  {k}: {v:.2f}%")

In [ ]:
# Cell 6: Query Alignment Analysis (KEY METRIC for v4!)

print("=" * 70)
print("QUERY ALIGNMENT ANALYSIS (v4 Key Metric)")
print("=" * 70)

if 'T_feat' in embeddings and 'Q_brep_self' in embeddings:
    T_feat = embeddings['T_feat']
    Q_brep_self = embeddings['Q_brep_self']
    Q_pc_self = embeddings.get('Q_pc_self')
    conf_text = embeddings.get('conf_text')
    
    # Normalize for cosine similarity
    T_feat_norm = F.normalize(T_feat, dim=-1)
    Q_brep_norm = F.normalize(Q_brep_self, dim=-1)
    
    # Per-slot cosine similarity
    cos_brep = (T_feat_norm * Q_brep_norm).sum(dim=-1)  # (N, K)
    
    # Weighted by confidence if available
    if conf_text is not None:
        weighted_cos_brep = (cos_brep * conf_text).sum(dim=1) / (conf_text.sum(dim=1) + 1e-8)
        query_align_brep = weighted_cos_brep.mean().item()
    else:
        query_align_brep = cos_brep.mean().item()
    
    print(f"\nBRep Query Alignment: {query_align_brep:.4f}")
    print(f"  Per-slot mean: {cos_brep.mean().item():.4f}")
    print(f"  Per-slot min: {cos_brep.min().item():.4f}")
    print(f"  Per-slot max: {cos_brep.max().item():.4f}")
    
    if Q_pc_self is not None:
        Q_pc_norm = F.normalize(Q_pc_self, dim=-1)
        cos_pc = (T_feat_norm * Q_pc_norm).sum(dim=-1)
        if conf_text is not None:
            weighted_cos_pc = (cos_pc * conf_text).sum(dim=1) / (conf_text.sum(dim=1) + 1e-8)
            query_align_pc = weighted_cos_pc.mean().item()
        else:
            query_align_pc = cos_pc.mean().item()
        print(f"\nPC Query Alignment: {query_align_pc:.4f}")
    
    # Assessment
    print("\n" + "-" * 50)
    if query_align_brep >= 0.70:
        print("✓ EXCELLENT: Query alignment achieves target (≥ 0.70)")
        print("  This means Q_self matches T_feat semantically!")
    elif query_align_brep >= 0.50:
        print("○ GOOD: Query alignment improving (0.50-0.70)")
    else:
        print(f"✗ NEEDS WORK: Query alignment below target (current: {query_align_brep:.4f}, target: ≥ 0.70)")
else:
    print("\nQuery features not available in model outputs.")
    query_align_brep = 0.0

In [ ]:
# Cell 7: Self-Grounding Quality

print("=" * 70)
print("SELF-GROUNDING QUALITY")
print("=" * 70)

# Cosine similarity between guided and self embeddings
cos_brep = (z_brep_guided * z_brep_self).sum(dim=-1).mean().item()
cos_pc = (z_pc_guided * z_pc_self).sum(dim=-1).mean().item()

print(f"\nBRep Guided-Self Cosine: {cos_brep:.4f}")
print(f"PC Guided-Self Cosine: {cos_pc:.4f}")

# Distribution
cos_brep_all = (z_brep_guided * z_brep_self).sum(dim=-1)
cos_pc_all = (z_pc_guided * z_pc_self).sum(dim=-1)

print(f"\nBRep Cosine Distribution:")
print(f"  Min: {cos_brep_all.min().item():.4f}")
print(f"  Max: {cos_brep_all.max().item():.4f}")
print(f"  Std: {cos_brep_all.std().item():.4f}")

# Assessment
print("\n" + "-" * 50)
if cos_brep >= 0.85:
    print("✓ EXCELLENT: Self-grounding achieves target (≥ 0.85)")
elif cos_brep >= 0.70:
    print("○ GOOD: Self-grounding improving (0.70-0.85)")
else:
    print(f"✗ NEEDS WORK: Self-grounding below target (current: {cos_brep:.4f}, target: ≥ 0.80)")

In [ ]:
# Cell 8: Text-Free Inference (Self-Grounding Retrieval)

print("=" * 70)
print("TEXT-FREE INFERENCE (Self-Grounding)")
print("=" * 70)

# Text → BRep with self-grounded embeddings
text_brep_self = compute_retrieval_metrics(z_text, z_brep_self)
print(f"\nText → BRep (Self-Grounding):")
for k, v in text_brep_self.items():
    print(f"  {k}: {v:.2f}%")

# Text → PC with self-grounded embeddings
text_pc_self = compute_retrieval_metrics(z_text, z_pc_self)
print(f"\nText → PC (Self-Grounding):")
for k, v in text_pc_self.items():
    print(f"  {k}: {v:.2f}%")

# Compare with guided
print("\n" + "-" * 50)
print("Comparison: Self vs Guided")
print("-" * 50)

r1_brep_diff = text_brep_self['R@1'] - text_brep_guided['R@1']
r1_pc_diff = text_pc_self['R@1'] - text_pc_guided['R@1']

print(f"Text→BRep R@1: Guided={text_brep_guided['R@1']:.2f}%, Self={text_brep_self['R@1']:.2f}%, Gap={abs(r1_brep_diff):.2f}%")
print(f"Text→PC R@1:   Guided={text_pc_guided['R@1']:.2f}%, Self={text_pc_self['R@1']:.2f}%, Gap={abs(r1_pc_diff):.2f}%")

if abs(r1_brep_diff) < 10:
    print("\n✓ Self-grounding achieves similar performance to text-guided (gap < 10%)!")
elif abs(r1_brep_diff) < 20:
    print("\n○ Self-grounding is close to text-guided (gap < 20%)")
else:
    print(f"\n△ Self-grounding needs improvement (gap: {abs(r1_brep_diff):.1f}%)")

In [ ]:
# Cell 9: Summary Table

print("\n" + "=" * 80)
print("EVALUATION SUMMARY - GFA v4")
print("=" * 80)

print(f"\n{'Metric':<35} {'Value':<15} {'Target':<15} {'Status'}")
print("-" * 80)

# Use query_align_brep if computed, else 0
q_align = query_align_brep if 'T_feat' in embeddings else 0.0

metrics = [
    ('Query Alignment (KEY!)', f"{q_align:.4f}", '≥0.70', q_align >= 0.70),
    ('Self-cos BRep', f"{cos_brep:.4f}", '≥0.80', cos_brep >= 0.80),
    ('Self-cos PC', f"{cos_pc:.4f}", '≥0.80', cos_pc >= 0.80),
    ('Text→BRep R@1 (guided)', f"{text_brep_guided['R@1']:.2f}%", '≥65%', text_brep_guided['R@1'] >= 65),
    ('Text→PC R@1 (guided)', f"{text_pc_guided['R@1']:.2f}%", '≥60%', text_pc_guided['R@1'] >= 60),
    ('Text→BRep R@1 (self)', f"{text_brep_self['R@1']:.2f}%", '≥55%', text_brep_self['R@1'] >= 55),
    ('Text→PC R@1 (self)', f"{text_pc_self['R@1']:.2f}%", '≥50%', text_pc_self['R@1'] >= 50),
    ('Gap (guided-self)', f"{abs(r1_brep_diff):.2f}%", '<10%', abs(r1_brep_diff) < 10),
]

for metric, value, target, passed in metrics:
    status = '✓' if passed else '✗'
    print(f"{metric:<35} {value:<15} {target:<15} {status}")

passed_count = sum(1 for _, _, _, p in metrics if p)
print(f"\nPassed: {passed_count}/{len(metrics)}")

# Overall assessment
print("\n" + "-" * 80)
if passed_count == len(metrics):
    print("✓ ALL TARGETS MET - v4 query distillation working as expected!")
elif passed_count >= len(metrics) - 2:
    print("○ MOSTLY PASSING - Minor adjustments may help")
elif q_align >= 0.70:
    print("△ Query alignment good - Retrieval needs tuning")
elif cos_brep >= 0.70:
    print("△ Self-grounding improving - Continue training")
else:
    print("✗ Self-grounding still needs work - Check query distillation")

In [ ]:
# Cell 10: Embedding Visualization (UMAP)

try:
    from umap import UMAP
    
    # Sample for visualization
    sample_size = min(1000, len(z_text))
    idx = np.random.choice(len(z_text), sample_size, replace=False)
    
    # Combine embeddings
    all_emb = np.vstack([
        z_text[idx].numpy(),
        z_brep_guided[idx].numpy(),
        z_brep_self[idx].numpy(),
    ])
    
    labels = np.array(
        ['Text'] * sample_size + 
        ['BRep (Guided)'] * sample_size + 
        ['BRep (Self)'] * sample_size
    )
    
    # Reduce dimensions
    print("Running UMAP dimensionality reduction...")
    reducer = UMAP(n_neighbors=15, min_dist=0.1, metric='cosine', random_state=42)
    emb_2d = reducer.fit_transform(all_emb)
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    # Plot 1: All three modalities
    ax = axes[0]
    colors = {'Text': 'blue', 'BRep (Guided)': 'green', 'BRep (Self)': 'orange'}
    for label in ['Text', 'BRep (Guided)', 'BRep (Self)']:
        mask = labels == label
        ax.scatter(emb_2d[mask, 0], emb_2d[mask, 1], 
                   c=colors[label], label=label, alpha=0.5, s=10)
    ax.legend()
    ax.set_title('Embedding Space (UMAP)')
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    
    # Plot 2: Guided vs Self overlap
    ax = axes[1]
    n_show = 50
    for i in range(n_show):
        guided_idx = sample_size + i
        self_idx = 2 * sample_size + i
        ax.plot([emb_2d[guided_idx, 0], emb_2d[self_idx, 0]], 
                [emb_2d[guided_idx, 1], emb_2d[self_idx, 1]], 
                'gray', alpha=0.3, linewidth=0.5)
    
    for label in ['BRep (Guided)', 'BRep (Self)']:
        mask = labels == label
        ax.scatter(emb_2d[mask, 0], emb_2d[mask, 1], 
                   c=colors[label], label=label, alpha=0.6, s=15)
    ax.legend()
    ax.set_title(f'Guided vs Self (Cosine: {cos_brep:.3f})')
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    
    plt.tight_layout()
    
    # Save figure
    output_path = Path('../outputs/gfa_v4')
    output_path.mkdir(parents=True, exist_ok=True)
    plt.savefig(output_path / 'embedding_visualization.png', dpi=150)
    print(f"Saved to {output_path / 'embedding_visualization.png'}")
    plt.show()
    
except ImportError:
    print("UMAP not installed. Run: pip install umap-learn")

In [ ]:
# Cell 11: Export Results to JSON

import json
from datetime import datetime

results = {
    'model': 'CLIP4CAD-GFA v4',
    'checkpoint': str(CHECKPOINT_PATH),
    'eval_date': datetime.now().isoformat(),
    'num_samples': len(val_dataset),
    'retrieval': {
        'text_brep_guided': text_brep_guided,
        'text_brep_self': text_brep_self,
        'text_pc_guided': text_pc_guided,
        'text_pc_self': text_pc_self,
        'pc_brep_guided': pc_brep_guided,
    },
    'self_grounding': {
        'cos_brep': cos_brep,
        'cos_pc': cos_pc,
    },
    'query_alignment': {
        'brep': q_align if 'T_feat' in embeddings else None,
    },
    'targets_met': {
        'query_align_target': q_align >= 0.70 if 'T_feat' in embeddings else False,
        'cos_brep_target': cos_brep >= 0.80,
        'self_retrieval_target': text_brep_self['R@1'] >= 55,
        'gap_target': abs(r1_brep_diff) < 10,
    }
}

output_path = Path('../outputs/gfa_v4')
output_path.mkdir(parents=True, exist_ok=True)

with open(output_path / 'eval_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"Results saved to {output_path / 'eval_results.json'}")
print("\nDone!")